# 🧠 การเรียนรู้แบบถ่ายโอน (Transfer Learning): การแช่แข็งโครงสร้างหลัก (Backbone Freezing) และการวิเคราะห์พารามิเตอร์

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Transfer Learning**! ในสมุดบันทึกนี้ เราจะ:
1. อธิบายทฤษฎีการนำคุณลักษณะกลับมาใช้ใหม่ (feature reuse) และลำดับขั้นของการถ่ายโอนความรู้ระหว่างเลเยอร์ (hierarchy of layer transferability)
2. โหลดโครงข่ายประสาทเทียมมาตรฐาน (`ResNet18`) ซึ่งเป็นโมเดลที่ผ่านการฝึกฝนล่วงหน้า (pre-trained model)
3. เขียนฟังก์ชันนับจำนวนพารามิเตอร์เพื่อวัดความซับซ้อนของโมเดล
4. พัฒนาการ **แช่แข็งน้ำหนักของโครงสร้างหลัก (backbone weight freezing)** โดยการกำหนดค่าการติดตามเกรเดียนต์ให้เป็น `False`
5. ปรับเปลี่ยนส่วนหัวลัพธ์ (output head) ของโมเดลเพื่อให้เหมาะสมกับงานจำแนกประเภทข้อมูลใหม่ที่กำหนดเอง
6. เปรียบเทียบความแตกต่างของจำนวนพารามิเตอร์ที่สามารถฝึกฝนได้ (trainable parameters) ก่อนและหลังการแช่แข็งในขั้นตอนการเรียนรู้แบบถ่ายโอน
7. เชื่อมโยงกระบวนการนี้เข้ากับการเริ่มต้นโมเดลด้วยค่าที่ฝึกฝนล่วงหน้าเป็นค่าเริ่มต้นของ YOLO และอาร์กิวเมนต์การฝึกฝน `freeze` ของ YOLO

เรามาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันเลยครับ

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# Set seed for reproducibility
torch.manual_seed(42)

## 1. การสร้างฟังก์ชันนับจำนวนพารามิเตอร์

เราจะเขียนฟังก์ชันช่วยในการคำนวณดังนี้:
-   **พารามิเตอร์ทั้งหมด (Total Parameters):** จำนวนน้ำหนัก (weights) และไบแอส (biases) ทั้งหมดที่มีอยู่ในโมเดล
-   **พารามิเตอร์ที่สามารถฝึกฝนได้ (Trainable Parameters):** พารามิเตอร์ที่จะถูกคำนวณค่าเกรเดียนต์ (ระบุ `requires_grad=True`) ซึ่งหมายความว่าจะถูกปรับปรุงค่าโดยออปติไมเซอร์ (optimizer)

In [ ]:
def count_parameters(model):
    """
    Count total and trainable parameters in a PyTorch model.
    """
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

## 2. การโหลดโครงสร้างหลัก ResNet18

เราจะสร้างออบเจกต์โมเดล `ResNet18` มาตรฐานขึ้นมา ซึ่งตามค่าเริ่มต้นแล้ว ทุกเลเยอร์จะสามารถฝึกฝนได้

In [ ]:
model = models.resnet18()

total_p, trainable_p = count_parameters(model)
print("--- Initial ResNet18 model ---")
print(f"Total Parameters    : {total_p:,}")
print(f"Trainable Parameters: {trainable_p:,}")

## 3. การประยุกต์ใช้การเรียนรู้แบบถ่ายโอน (การแช่แข็งและการเปลี่ยนส่วนหัวลัพธ์)

ตอนนี้เราจะนำโมเดลนี้กลับมาใช้ใหม่สำหรับงานเป้าหมายที่มี **10 คลาสที่กำหนดเอง (10 custom classes)**:
1.  **แช่แข็งโครงสร้างหลัก (Freeze Backbone):** ทำการวนลูปผ่านพารามิเตอร์ทั้งหมดและปิดการคำนวณเกรเดียนต์ (`requires_grad = False`) ซึ่งจะแช่แข็งตัวสกัดคุณลักษณะ (feature extractor) เอาไว้
2.  **เปลี่ยนส่วนหัวลัพธ์ (Replace Output Head):** แทนที่เลเยอร์เชื่อมต่อเต็มรูปแบบสุดท้าย (final fully connected layer หรือ `model.fc`) ด้วยเลเยอร์เชิงเส้น (linear layer) ใหม่ ซึ่งเลเยอร์ใหม่นี้จะกำหนดค่า `requires_grad = True` ให้โดยอัตโนมัติตามค่าเริ่มต้น

In [ ]:
# Step 1: Freeze all parameters
for param in model.parameters():
    param.requires_grad = False

total_p, trainable_p = count_parameters(model)
print("--- After Freezing Feature Extractor ---")
print(f"Total Parameters    : {total_p:,}")
print(f"Trainable Parameters: {trainable_p:,}")

# Step 2: Replace final fully connected layer
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 10)

total_p, trainable_p = count_parameters(model)
print("\n--- After Replacing Output Head (10 Classes) ---")
print(f"Total Parameters    : {total_p:,}")
print(f"Trainable Parameters: {trainable_p:,}")

สังเกตความเปลี่ยนแปลงกันครับ:
-   **พารามิเตอร์ทั้งหมด (Total Parameters):** คงเดิมแทบจะไม่เปลี่ยนแปลง (ประมาณ $11.2$ ล้านพารามิเตอร์)
-   **พารามิเตอร์ที่สามารถฝึกฝนได้ (Trainable Parameters):** ลดลงอย่างมหาศาลจาก $11,176,512$ พารามิเตอร์ เหลือเพียงแค่ **$5,130$** พารามิเตอร์ (ซึ่งเป็นค่าน้ำหนัก $512 \times 10$ ตัว $+$ ไบแอส 10 ตัวในเลเยอร์สุดท้าย)
-   **ประโยชน์ที่ได้รับ:** การฝึกฝนโมเดลจะทำได้รวดเร็วขึ้นอย่างมากและใช้หน่วยความจำน้อยลง เนื่องจากจะคำนวณเกรเดียนต์เฉพาะเลเยอร์เอาต์พุตสุดท้ายเท่านั้น ในขณะที่ตัวสกัดคุณลักษณะที่ผ่านการฝึกฝนล่วงหน้าจะยังคงเสถียรไม่มีการเปลี่ยนแปลง

## 💡 การเชื่อมโยงกับ YOLO และการเรียนรู้เชิงลึก (Deep Learning)
*   **โมเดลที่ผ่านการฝึกฝนล่วงหน้าตามค่าเริ่มต้น (Default Pre-training):** เมื่อคุณสั่งรันคำสั่ง `yolo train model=yolo11n.pt` ทาง YOLO จะทำกระบวนการถ่ายโอนการเรียนรู้ให้โดยอัตโนมัติ โดยการโหลดน้ำหนักที่ฝึกฝนล่วงหน้ามาจากชุดข้อมูล COCO เพื่อกำหนดค่าเริ่มต้นให้กับเครือข่ายประสาทเทียม
*   **อาร์กิวเมนต์ `freeze`:** หากคุณต้องการแช่แข็งเลเยอร์โครงสร้างหลัก (backbone layers) ใน YOLO เพื่อป้องกันไม่ให้น้ำหนักเปลี่ยนแปลง คุณสามารถตั้งค่าผ่านพารามิเตอร์ `freeze` ได้ดังนี้:
    ```python
    from ultralytics import YOLO
    model = YOLO('yolo11n.pt')
    model.train(data='data.yaml', epochs=50, freeze=10) # Freezes the first 10 layers
    ```
    วิธีนี้มีประโยชน์เป็นอย่างยิ่งเมื่อต้องทำการปรับจูนละเอียด (fine-tuning) บนชุดข้อมูลที่มีขนาดเล็กมาก เพื่อป้องกันไม่ให้เกิดการเรียนรู้ดีเกินไป (overfitting)